In [43]:
!pip install -qU google-generativeai==0.8.5 google-ai-generativelanguage==0.6.15 langgraph langchain langchain-google-genai openai

#import api key


In [44]:
import os
import getpass
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

In [45]:
os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter the Gemini API Key")

Enter the Gemini API Key··········


In [46]:
llm = ChatGoogleGenerativeAI(model = "models/gemini-2.5-flash", temperature =0.3)

#Nodes

#Get Get user input

In [47]:
def get_input(state: dict) -> dict:
    user_input = input("Hi, how are you feeling today?\n")
    state["text"] = user_input
    return state

#Classify mental health condition

In [48]:
def classify(state: dict) -> dict:
    prompt = f"""
    Classify the user's mental condition into one:
    - Normal
    - Stress
    - Anxiety
    - Depression
    - Crisis

    Text: {state['text']}

    Only return one word.
    """

    response = llm.invoke([HumanMessage(content=prompt)])
    category = response.content.strip()
    print("Detected:", category)

    state["category"] = category
    return state


#create router

In [49]:
def symptom_router(state:dict) -> dict:
      cat = state["category"].lower()
      if "stress" in cat:
       return "stress"
      elif "anxiety" in cat:
        return "anxiety"
      elif "depression" in cat:
        return "depression"
      elif "crisis" in cat:
        return "crisis"
      else:
        return "normal"



#Responses

In [50]:
def normal_node(state: dict) -> dict:
  state["answer"] = "You seem okay Keep taking care of yourself!"
  return state

def stress_node(state:dict) -> dict:
  state["answer"] = "You might be stressed. Try deep breathing, rest, or talk to a friend."
  return state

def anxiety_node(state: dict) -> dict:
   state["answer"] = "It looks like anxiety. Try slow breathing and grounding techniques."
   return state

def depression_node(state: dict) -> dict:
   state["answer"] = "You may be feeling low. Please talk to someone you trust "
   return state

def crisis_node(state: dict) -> dict:
   state["answer"] = "This looks serious. Please seek immediate help from a doctor or helpline "
   return state


#Build LangGraph

In [51]:
builder = StateGraph(dict)

builder.set_entry_point("input")

builder.add_node("input", get_input)
builder.add_node("classify", classify)
builder.add_node("normal", normal_node)
builder.add_node("stress", stress_node)
builder.add_node("anxiety", anxiety_node)
builder.add_node("depression", depression_node)
builder.add_node("crisis", crisis_node)

builder.add_edge("input", "classify")

builder.add_conditional_edges("classify", symptom_router, {
    "normal": "normal",
    "stress": "stress",
    "anxiety": "anxiety",
    "depression": "depression",
    "crisis": "crisis"
})

builder.add_edge("normal", END)
builder.add_edge("stress", END)
builder.add_edge("anxiety", END)
builder.add_edge("depression", END)
builder.add_edge("crisis", END)


#Compile and invoke the graph

In [52]:
graph = builder.compile()

In [ ]:
final_state = graph.invoke({})
print("final Output \n")
print(final_state["answer"])